# Modeling: Spaceship Titanic

Author: Finian O'Neill
Purpose: Turn the EDA findings (`eda.ipynb`, section 15) into a validated model and a submission.

**Contents**
1. Setup
2. Preprocessing: imputation rules and features
3. Validation setup
4. Baseline: logistic regression
5. CatBoost
6. Ablation: do the EDA features help?
7. Feature importance
8. Results
9. Submission
10. Notes & next steps (round 1)

**Round 2:** closing the gap to the leaderboard
11. Multi-seed baseline
12. Imputation: how to fill spending
13. Cabin neighbor feature
14. LightGBM & XGBoost
15. Ensemble
16. Optuna tuning
17. Round 2 results & submission
18. Notes (round 2)

**Round 3:** new information
19. Group & cabin-neighborhood features
20. Pseudo-labeling
21. Round 3 results & submission
22. Notes (round 3)

Preprocessing lives in `app/preprocessing.py` and the CV and model helpers in `app/modeling.py`, so this notebook only records experiments.

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from app import preprocessing as pp
from app import modeling as mdl

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
plt.style.use('fivethirtyeight')

## 2. Preprocessing: imputation rules and features
The deterministic rules from the EDA run first, then a statistical fallback using train-only statistics. The log shows how many values each step filled.

In [ ]:
train_df = pd.read_csv('data/train.csv')
test_df = pd.read_csv('data/test.csv')

full_df, imputation_log = pp.preprocess(train_df, test_df)
imputation_log

In [ ]:
# The EDA rules still hold after imputation
print('CryoSleep passengers with spend:', int((full_df.CryoSleep & (full_df.TotalSpending > 0)).sum()))
print('Children (<13) with spend:      ', int(((full_df.Age < 13) & (full_df.TotalSpending > 0)).sum()))

X, y, groups, X_test, test_ids = pp.split_xy(full_df)
print(f'\nX: {X.shape}, X_test: {X_test.shape}')
print(f'Categorical ({len(pp.CATEGORICAL_FEATURES)}): {pp.CATEGORICAL_FEATURES}')
print(f'Numeric ({len(pp.NUMERIC_FEATURES)}): {pp.NUMERIC_FEATURES}')

## 3. Validation setup
`StratifiedGroupKFold` (5 folds, fixed seed) on `GroupId`, so groupmates never land on both sides of a split. Every experiment uses the same splits, so their scores can be compared directly.

In [ ]:
splits = mdl.cv_splits(y, groups)
for i, (tr_idx, va_idx) in enumerate(splits):
    shared = set(groups.iloc[tr_idx]) & set(groups.iloc[va_idx])
    print(f'Fold {i}: train {len(tr_idx)}, valid {len(va_idx)}, '
          f'valid target rate {y.iloc[va_idx].mean():.3f}, shared groups {len(shared)}')

results = []

## 4. Baseline: logistic regression
One-hot categoricals and scaled numerics. Because it's linear, the gap between it and the tree models shows how much the interactions are worth.

In [ ]:
results.append(mdl.cross_validate(
    mdl.logreg_fit_predict(pp.CATEGORICAL_FEATURES), X, y, splits, name='LogReg (all features)'))

## 5. CatBoost
Native categorical handling, default-ish parameters (1,000 iterations, lr 0.03, depth 6). No tuning yet.

In [ ]:
cb_all = mdl.cross_validate(
    mdl.catboost_fit_predict(pp.CATEGORICAL_FEATURES), X, y, splits, name='CatBoost (all features)')
results.append(cb_all)

## 6. Ablation: do the EDA features help?
Drop each engineered feature (or group) and re-run CatBoost on the same splits. If the score drops, the feature helps.

Each fold's accuracy has a standard deviation of about 0.012 across folds, so differences in the mean of a few thousandths are within noise *on their own*. Because every run uses the **same splits**, the fold-by-fold comparison is more sensitive: a feature that lowers accuracy in 5 of 5 folds when dropped is very likely real.

In [ ]:
ablations = {
    'LuxuryShare': ['LuxuryShare'],
    'CryoSleep_Deck': ['CryoSleep_Deck'],
    'Deck_Side': ['Deck_Side'],
    'Route': ['Route'],
    'all 4 interactions': ['LuxuryShare', 'CryoSleep_Deck', 'Deck_Side', 'Route'],
    'spend aggregates': ['log_TotalSpending', 'log_LuxurySpending', 'log_BasicSpending',
                         'HasSpent', 'NumAmenitiesUsed', 'LuxuryShare'],
}
ablation_results = []
for label, drop in ablations.items():
    cols = [c for c in X.columns if c not in drop]
    r = mdl.cross_validate(mdl.catboost_fit_predict(pp.CATEGORICAL_FEATURES), X[cols], y, splits,
                           name=f'CatBoost − {label}')
    r['delta'] = r['cv_mean'] - cb_all['cv_mean']
    r['folds_lower'] = int((np.array(r['folds']) < np.array(cb_all['folds'])).sum())
    ablation_results.append(r)

pd.DataFrame([{'Dropped': r['name'].replace('CatBoost − ', ''), 'CV mean': r['cv_mean'],
               'Δ vs all features': r['delta'], 'Folds lower (of 5)': r['folds_lower']}
              for r in ablation_results]).set_index('Dropped').round(4)

## 7. Feature importance
CatBoost `PredictionValuesChange` importance, from a model fit on the full training set.

In [ ]:
cb_full = mdl.catboost_fit_predict(pp.CATEGORICAL_FEATURES)
test_proba_cb = cb_full(X, y, X_test)
importance = (pd.Series(cb_full.last_model.get_feature_importance(), index=X.columns)
                .sort_values(ascending=True))
plt.figure(figsize=(9, 8))
importance.plot.barh(color='#1b9e77')
plt.title('CatBoost feature importance (full train)', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Results

In [ ]:
table = mdl.results_table(results + ablation_results)
table.sort_values('CV mean', ascending=False)

In [ ]:
# Out-of-fold accuracy by key segment, to see where the model is weakest
oof_pred = (cb_all['oof'] >= 0.5).astype(int)
seg = full_df[full_df.Dataset == 'train'].reset_index(drop=True).assign(correct=(oof_pred == y.values))
seg['Segment'] = np.select(
    [seg.CryoSleep, seg.HasSpent == 1],
    ['CryoSleep', 'Awake, spender'], default='Awake, non-spender')
seg.groupby('Segment').agg(N=('correct', 'size'), OOF_accuracy=('correct', 'mean'),
                           Transported_rate=('Transported', 'mean')).round(3)

## 9. Submission
The CatBoost model fit on the full training set (section 7), using a 0.5 threshold. Files go to `submissions/`, which git ignores.

In [ ]:
sub_path = mdl.write_submission(test_ids, test_proba_cb, 'catboost_all', cb_all['cv_mean'])
pd.read_csv(sub_path).head()

To submit (from the repo root, with the env active):
```bash
kaggle competitions submit -c spaceship-titanic -f <path above> -m "CatBoost all features, CV <score>"
```
Then record the public leaderboard score in section 10.

## 10. Notes & next steps

> `model.ipynb` outputs are stripped on commit (nbstripout), so the key numbers are recorded here.

**Results (5-fold StratifiedGroupKFold on GroupId, seed 42)**

| Experiment | CV accuracy | Public LB |
|---|---|---|
| LogReg, all 24 features | 0.7990 ± 0.0161 | – |
| **CatBoost, all 24 features** | **0.8164 ± 0.0121** | **0.80266** (2026-11-21, ~rank 1,060 of 2,161) |

**Leaderboard context:** 2,161 teams, median 0.8024. The board is extremely compressed: 512 teams sit between 0.800 and 0.805, ≥ 0.81 is the top 11%, and ≥ 0.82 the top 5%. The public score's standard error is about ±0.8 points, so rank swings by hundreds within noise. **Make decisions on CV, not leaderboard rank.**

**What we learned**
- **Trees beat the linear model by about 1.7 points.** The interactions matter, and CatBoost's native categorical handling makes it a strong single-model baseline.
- **The EDA features help.** Dropping all four interaction features costs 0.45 points and lowers accuracy in 5 of 5 folds. `Deck_Side` alone costs 0.41 points (5 of 5 folds). `LuxuryShare`, `CryoSleep_Deck` and `Route` each cost about 0.2 points (4 of 5 folds). They're small one at a time but consistent, so **keep all of them**.
- **`LuxuryShare` is CatBoost's most important feature** by a wide margin, followed by `Cabin_num`, `HomePlanet` and `Deck`. That matches the EDA. `VIP_int`, `IsSolo` and `HasSpent` contribute almost nothing (`HasSpent` overlaps with `LuxuryShare` and the log spending columns).
- **The weakest segment is awake non-spenders** (532 passengers, 71.8% out-of-fold accuracy, mostly children). Sleepers are predicted at 81.6% accuracy, which is about the most possible given that sleepers on decks E and G are only 65% transported.

**Next steps**
1. **Submit** `submissions/catboost_all_cv0.8164.csv` and record the public leaderboard score above. If it's more than about 1 point off CV, check the validation setup before going further.
2. **LightGBM and XGBoost** on the same splits, then **average** the out-of-fold probabilities (the out-of-fold predictions are already stored in each `results` entry).
3. **Tune the threshold** on the out-of-fold predictions. The split is roughly 50/50, so expect small gains.
4. **Optuna** on CatBoost (depth, l2_leaf_reg, learning_rate and iterations, with early stopping inside each fold), about 50 trials. Do this only after the feature set is fixed.
5. **Features to try:** whether groupmates were asleep (`GroupCryoShare`), and finer age bands for children (the weakest segment). Drop `VIP_int` and `IsSolo` if an ablation confirms they're noise.

# Round 2: closing the gap to the leaderboard

Round 1 scored 0.8164 in CV (one split) and **0.80266 on the public leaderboard**. Round 2 has three rules:
- **Every experiment runs on the same 5 split seeds** (`mdl.EVAL_SEEDS`). Results are compared *paired*, seed by seed, so differences of a few tenths of a point can be told apart from split noise.
- The metric is **OOF accuracy** (every train row predicted once per seed), averaged over seeds.
- Decisions are made in code by the rules stated in each section, not by eye. Optuna tunes on a separate split seed (`mdl.TUNING_SEED`) that is never used for evaluation.

## 11. Multi-seed baseline
The round-1 CatBoost on 5 split seeds. Every later experiment is compared to this.

In [ ]:
cat = pp.CATEGORICAL_FEATURES
nog = lambda df: df.drop(columns='GroupId')  # model frames without the group id

# Train/test frames for each spending-fill strategy (GroupId kept for fold-aware features)
frames = {}
for sf in ['median', 'zero', 'nan']:
    f, _ = pp.preprocess(train_df, test_df, spend_fill=sf, verbose=False)
    Xg, y_sf, g_sf, Xtg, _ = pp.split_xy(f, pp.FEATURES + ['GroupId'])
    assert y_sf.equals(y) and g_sf.equals(groups)  # same row order for every variant
    frames[sf] = (Xg, Xtg)

base_cb = mdl.multi_seed_cv(mdl.catboost_fit_predict(cat), nog(frames['median'][0]), y, groups,
                            'CatBoost, median spend fill (round 1)')
print(f"\nRound-1 single split: {cb_all['cv_mean']:.4f}. Seeds range "
      f"{base_cb['per_seed'].min():.4f}-{base_cb['per_seed'].max():.4f}: the choice of split alone moves CV this much.")

## 12. Imputation: how to fill spending
After the deterministic rules, 806 spending values are still missing (awake adults). Round 1 filled them with the awake-adult median by HomePlanet (Europa's FoodCourt median is 1,571), which directly sets `LuxuryShare`, the top feature. The alternatives: fill with **0**, or leave them as **NaN** and let CatBoost handle it (`LuxuryShare` and the totals then use the known values).

**Rule:** keep the fill with the best mean OOF accuracy.

In [ ]:
spend_results = {'median': base_cb}
for sf in ['zero', 'nan']:
    spend_results[sf] = mdl.multi_seed_cv(mdl.catboost_fit_predict(cat), nog(frames[sf][0]), y, groups,
                                          f'CatBoost, spend fill = {sf}')
best_fill = max(spend_results, key=lambda k: spend_results[k]['mean'])
print(f'\n-> Using spend_fill = {best_fill!r}')
mdl.comparison_table(list(spend_results.values()), base_cb)

## 13. Cabin neighbor feature
`NeighborRate` is the smoothed transported rate of passengers within ±k cabins on the same deck and side, **excluding the passenger's own group**. It's computed inside each fold from training-fold labels only (`mdl.with_neighbor_feature`). On shuffled labels it carries no signal (corr 0.007 vs 0.22 on the real labels), which confirms there's no leakage. Try k = 5, 10 and 25.

**Rule:** adopt the best k only if it beats the section 12 winner on mean *and* on at least 4 of 5 seeds.

In [ ]:
X_best, Xt_best = frames[best_fill]
cb_ref = spend_results[best_fill]
nbr_results = {k: mdl.multi_seed_cv(mdl.with_neighbor_feature(mdl.catboost_fit_predict(cat), k=k),
                                    X_best, y, groups, f'CatBoost + NeighborRate (k={k})')
               for k in [5, 10, 25]}
best_k = max(nbr_results, key=lambda k: nbr_results[k]['mean'])
d = nbr_results[best_k]['per_seed'] - cb_ref['per_seed']
use_nbr = bool(d.mean() > 0 and (d > 0).sum() >= 4)
print(f'\n-> Best k={best_k}: Δ {d.mean():+.4f}, better on {(d > 0).sum()}/5 seeds -> '
      f'{"ADOPT" if use_nbr else "REJECT"} NeighborRate')

wrap = (lambda fp: mdl.with_neighbor_feature(fp, k=best_k)) if use_nbr else (lambda fp: fp)
X_model, Xt_model = (X_best, Xt_best) if use_nbr else (nog(X_best), nog(Xt_best))
cb_current = nbr_results[best_k] if use_nbr else cb_ref
mdl.comparison_table([cb_ref] + list(nbr_results.values()), cb_ref)

## 14. LightGBM & XGBoost
Same preprocessing and features, with reasonable default parameters. The question is whether they're good enough, *and different enough*, to help in an ensemble.

In [ ]:
lgb_r = mdl.multi_seed_cv(wrap(mdl.lgbm_fit_predict(cat)), X_model, y, groups, 'LightGBM')
xgb_r = mdl.multi_seed_cv(wrap(mdl.xgb_fit_predict(cat)), X_model, y, groups, 'XGBoost')

oof_corr = np.corrcoef([cb_current['oofs'][0], lgb_r['oofs'][0], xgb_r['oofs'][0]])
pd.DataFrame(oof_corr, index=['CatBoost', 'LightGBM', 'XGBoost'],
             columns=['CatBoost', 'LightGBM', 'XGBoost']).round(3)

## 15. Ensemble
Equal-weight average of out-of-fold probabilities, seed by seed.

In [ ]:
ens = mdl.score_result(mdl.ensemble_result([cb_current, lgb_r, xgb_r], 'Ensemble: CB + LGBM + XGB'), y)
ens_cx = mdl.score_result(mdl.ensemble_result([cb_current, xgb_r], 'Ensemble: CB + XGB'), y)
mdl.comparison_table([cb_current, lgb_r, xgb_r, ens, ens_cx], base_cb)

## 16. Optuna tuning
40 trials of TPE over depth, learning rate, iterations, L2, random strength and bagging temperature, on the **tuning split seed** (different from the evaluation seeds). The tuned parameters are then re-evaluated on the 5 evaluation seeds, which gives an honest estimate rather than the optimistic best-trial score.

In [ ]:
study = mdl.tune_catboost(X_model, y, groups, cat, n_trials=40, wrap=wrap if use_nbr else None)
best_params = study.best_params
print(f'Best trial on tuning split: {study.best_value:.4f}')
print(best_params)

cb_tuned = mdl.multi_seed_cv(wrap(mdl.catboost_fit_predict(cat, **best_params)), X_model, y, groups,
                             'CatBoost (tuned)')
ens_tuned = mdl.score_result(mdl.ensemble_result([cb_tuned, lgb_r, xgb_r], 'Ensemble: tuned CB + LGBM + XGB'), y)

study.trials_dataframe().sort_values('value', ascending=False).head(8)[
    ['value', 'params_depth', 'params_learning_rate', 'params_iterations', 'params_l2_leaf_reg',
     'params_random_strength', 'params_bagging_temperature']].round(4)

## 17. Round 2 results & submission
**Rule:** pick the best candidate by mean OOF accuracy. Write a submission only if it beats the round-1 baseline by **at least 0.3 points**. Below that, the public leaderboard's noise (about ±0.8 points) makes a submission uninformative.

The final model is refit on all of train with the same 5 model seeds, and the test probabilities are averaged.

In [ ]:
summary = mdl.comparison_table(
    [base_cb, spend_results['zero'], spend_results['nan'], *nbr_results.values(),
     lgb_r, xgb_r, ens, ens_cx, cb_tuned, ens_tuned], base_cb).sort_values('OOF acc', ascending=False)
summary

In [ ]:
factories = {
    'cb': lambda s: wrap(mdl.catboost_fit_predict(cat, random_seed=s)),
    'cb_tuned': lambda s: wrap(mdl.catboost_fit_predict(cat, random_seed=s, **best_params)),
    'lgbm': lambda s: wrap(mdl.lgbm_fit_predict(cat, random_state=s)),
    'xgb': lambda s: wrap(mdl.xgb_fit_predict(cat, random_state=s)),
}
candidates = {  # name -> (result, components, file slug)
    cb_current['name']: (cb_current, ['cb'], 'catboost_r2'),
    cb_tuned['name']: (cb_tuned, ['cb_tuned'], 'catboost_tuned'),
    ens['name']: (ens, ['cb', 'lgbm', 'xgb'], 'ensemble'),
    ens_cx['name']: (ens_cx, ['cb', 'xgb'], 'ensemble_cb_xgb'),
    ens_tuned['name']: (ens_tuned, ['cb_tuned', 'lgbm', 'xgb'], 'ensemble_tuned'),
}
best_name = max(candidates, key=lambda k: candidates[k][0]['mean'])
best, components, slug = candidates[best_name]
gain = best['mean'] - base_cb['mean']
d = best['per_seed'] - base_cb['per_seed']
print(f'Best: {best_name}  OOF acc {best["mean"]:.4f}  Δ vs round 1 {gain:+.4f} '
      f'(better on {(d > 0).sum()}/5 seeds)')

if gain >= 0.003:
    test_proba = mdl.fit_full_predict([factories[c] for c in components], X_model, y, Xt_model)
    sub_path_r2 = mdl.write_submission(test_ids, test_proba, slug, best['mean'])
    agree = (pd.read_csv(sub_path_r2).Transported == pd.read_csv(sub_path).Transported).mean()
    print(f'Agreement with round-1 submission: {agree:.1%}')
else:
    sub_path_r2 = None
    print('Gain < 0.3 points: not writing a submission.')

## 18. Notes (round 2)

**Result: we're at a plateau at about 0.817 OOF accuracy.** None of the five round 2 levers cleared the +0.3-point bar, so no new submission was made.

| Experiment | OOF acc (5 seeds) | Δ vs round 1 | Seeds better |
|---|---|---|---|
| Round 1 CatBoost (median spend fill) | **0.8169 ± 0.0012** | – | – |
| Spend fill = 0 | 0.8169 | 0.0000 | 3/5 |
| Spend fill = NaN | 0.8158 | −0.0011 | 1/5 |
| + NeighborRate (best k = 5) | 0.8143 | −0.0026 | 0/5 |
| LightGBM / XGBoost alone | 0.8133 / 0.8158 | −0.0036 / −0.0011 | 0/5 / 1/5 |
| Ensemble CB + XGB (best) | 0.8174 | +0.0005 | 3/5 |
| CatBoost tuned (Optuna, 40 trials) | 0.8150 | −0.0019 | 1/5 |

**What we learned**
1. **The round 1 CV wasn't luck.** Across 5 seeds it's 0.8169 ± 0.0012, and the original single split (0.8164) sits in the middle. So the 1.4-point drop to the public leaderboard (0.8027) isn't a CV artifact. It's either leaderboard noise (about 1.7 standard errors) or something about the public test rows that CV can't see.
2. **How spending is filled doesn't matter.** Median, 0, and NaN are all within ±0.1 points. The concern that the median fill was distorting `LuxuryShare` was unfounded, because only about 6% of rows are affected.
3. **`NeighborRate` hurts, consistently (0/5 seeds for every k).** Its 0.22 correlation with the target is already captured by `Deck`, `Side`, `Cabin_num` and `CryoSleep`. Adding a noisy target-derived version gives the trees another way to overfit. The rejection rule worked as intended.
4. **The three boosting libraries make nearly identical predictions.** Their OOF predictions correlate at 0.98–0.99, so averaging them adds almost nothing (+0.05 points). Ensembling only pays off when models disagree.
5. **Tuning overfit the tuning split.** The best Optuna trial scored 0.8166 on its own split, but only 0.8150 on the evaluation seeds, which is *worse* than the defaults. With about 8,700 rows, the objective is flat and noisy (the top 8 trials are within 0.12 points of each other), so the default CatBoost parameters are already near-optimal. Keeping a separate tuning seed is what exposed this.

**Implication:** with this feature set, the model is at the limit of what the information supports. Further gains need **new information**, not more modeling:
- **Pseudo-labeling:** add confidently predicted test rows (for example, sleepers on decks A–D and F, which are 94–99% transported) to training. Groups are entirely within train or entirely within test, so test passengers' own groupmates are only ever seen through pseudo-labels.
- **Group-level features from the combined data:** for example, the share of each group that's asleep, and group spending patterns, built from train and test together. These don't use the target.
- **The weakest segment:** awake non-spenders (72% OOF accuracy, mostly children). Finer age handling or group-context features for children.

**Decision rule kept:** a submission is only worthwhile when CV improves by ≥ 0.3 points across seeds. Below that, a leaderboard score mostly measures public-split noise.

# Round 3: new information

Round 2 showed the model is at its limit with the current features (about 0.817). Round 3 adds information instead of modeling:
- **Group features** (no target): what a passenger's *other* groupmates look like (share asleep, spending, luxury share, children, cabins shared, shared last names, age gap), plus `IsChild`.
- **Cabin-neighborhood features** (no target): `Cabin_Occupancy` (passengers in the same real cabin) and `Nbr_CryoShare` (share asleep within ±10 cabins on the same deck and side, excluding the passenger's own group).
- **Pseudo-labeling:** inside each fold, add confidently predicted test rows to training.

Same protocol as before: 5 paired seeds, compared against the round-2 CatBoost setup (spend fill = `best_fill`). **Rule:** adopt a change only if it beats the reference on mean *and* on at least 4 of 5 seeds.

## 19. Group & cabin-neighborhood features

In [ ]:
full_r3, _ = pp.preprocess(train_df, test_df, spend_fill=best_fill, verbose=False)
full_r3 = pp.add_group_features(full_r3)

feat_sets = {
    'base': pp.FEATURES,
    '+ group': pp.FEATURES + pp.GROUP_FEATURES,
    '+ neighborhood': pp.FEATURES + pp.NEIGHBORHOOD_FEATURES,
    '+ group + neighborhood': pp.FEATURES + pp.GROUP_FEATURES + pp.NEIGHBORHOOD_FEATURES,
}
r3_data = {}
for k, feats in feat_sets.items():
    Xk, yk, gk, Xtk, _ = pp.split_xy(full_r3, feats)
    assert yk.equals(y) and gk.equals(groups)
    r3_data[k] = (Xk, Xtk)

# Correlation of each new feature with the target (train rows)
tr3 = full_r3[full_r3.Dataset == 'train']
pd.Series({c: tr3[[c, 'Transported']].dropna().astype(float).corr().iloc[0, 1]
           for c in pp.GROUP_FEATURES + pp.NEIGHBORHOOD_FEATURES}, name='corr with Transported').round(3)

In [ ]:
r3_ref = mdl.multi_seed_cv(mdl.catboost_fit_predict(cat), r3_data['base'][0], y, groups,
                           'CatBoost, round-2 setup (reference)')
grp_results = {k: mdl.multi_seed_cv(mdl.catboost_fit_predict(cat), r3_data[k][0], y, groups, f'CatBoost {k}')
               for k in ['+ group', '+ neighborhood', '+ group + neighborhood']}

def adopt(res, ref):
    d = res['per_seed'] - ref['per_seed']
    return bool(d.mean() > 0 and (d > 0).sum() >= 4)

passing = [k for k, r in grp_results.items() if adopt(r, r3_ref)]
best_set = max(passing, key=lambda k: grp_results[k]['mean']) if passing else 'base'
print(f'\n-> Feature set: {best_set!r}' + ('' if passing else ' (no new feature set passed the rule)'))
mdl.comparison_table([r3_ref] + list(grp_results.values()), r3_ref)

In [ ]:
# Where do the new features rank? (CatBoost fit on all of train with every feature)
X_all = r3_data['+ group + neighborhood'][0]
cb_imp = mdl.catboost_fit_predict(cat)
_ = cb_imp(X_all, y, X_all.head(1))
imp = pd.Series(cb_imp.last_model.get_feature_importance(), index=X_all.columns).sort_values(ascending=False)
new_cols = set(pp.GROUP_FEATURES + pp.NEIGHBORHOOD_FEATURES)
imp.to_frame('importance').assign(new=lambda d: d.index.isin(new_cols)).head(15).round(2)

## 20. Pseudo-labeling
Inside each fold: the fold model predicts the test set, test rows with probability ≥ `hi` or ≤ `lo` are added with hard labels, and the model is retrained before predicting the validation fold. Validation labels are never used. Two confidence levels are tested, on the feature set chosen in section 19.

In [ ]:
X_r3, Xt_r3 = r3_data[best_set]
r3_cur = r3_ref if best_set == 'base' else grp_results[best_set]

pseudo_results = {}
for hi, lo in [(0.95, 0.05), (0.90, 0.10)]:
    fp = mdl.with_pseudo_labels(mdl.catboost_fit_predict(cat), Xt_r3, hi=hi, lo=lo)
    res = mdl.multi_seed_cv(fp, X_r3, y, groups, f'CatBoost {best_set} + pseudo ({hi}/{lo})')
    print(f'   pseudo rows per fold: {np.mean(fp.n_pseudo):.0f} of {len(Xt_r3)} '
          f'({np.nanmean(fp.pseudo_rate):.0%} labeled transported)')
    pseudo_results[(hi, lo)] = res

passing = [k for k, r in pseudo_results.items() if adopt(r, r3_cur)]
best_pseudo = max(passing, key=lambda k: pseudo_results[k]['mean']) if passing else None
print(f'\n-> Pseudo-labeling: {best_pseudo if best_pseudo else "REJECTED (no threshold passed the rule)"}')
mdl.comparison_table([r3_cur] + list(pseudo_results.values()), r3_cur)

## 21. Round 3 results & submission
Combine the adopted pieces and compare against the **round-1 baseline**. As in round 2, a submission file is written only if the gain is **≥ 0.3 points**.

In [ ]:
r3_best = pseudo_results[best_pseudo] if best_pseudo else r3_cur
gain = r3_best['mean'] - base_cb['mean']
d = r3_best['per_seed'] - base_cb['per_seed']
print(f'Round 3 best: {r3_best["name"]}  OOF acc {r3_best["mean"]:.4f}  '
      f'Δ vs round 1 {gain:+.4f} (better on {(d > 0).sum()}/5 seeds)')

summary_r3 = mdl.comparison_table(
    [base_cb, r3_ref, *grp_results.values(), *pseudo_results.values()], base_cb
).sort_values('OOF acc', ascending=False)

if gain >= 0.003:
    if best_pseudo:
        factory = lambda s: mdl.with_pseudo_labels(mdl.catboost_fit_predict(cat, random_seed=s), Xt_r3,
                                                   hi=best_pseudo[0], lo=best_pseudo[1])
    else:
        factory = lambda s: mdl.catboost_fit_predict(cat, random_seed=s)
    test_proba_r3 = mdl.fit_full_predict([factory], X_r3, y, Xt_r3)
    sub_path_r3 = mdl.write_submission(test_ids, test_proba_r3, 'catboost_r3', r3_best['mean'])
    agree = (pd.read_csv(sub_path_r3).Transported == pd.read_csv(sub_path).Transported).mean()
    print(f'Agreement with round-1 submission: {agree:.1%}')
else:
    sub_path_r3 = None
    print('Gain < 0.3 points: not writing a submission.')
summary_r3

## 22. Notes (round 3)

**Result: the plateau holds.** Neither new information source passed the adoption rule, and no submission was written.

| Experiment (5 paired seeds) | OOF acc | Δ vs reference | Seeds better |
|---|---|---|---|
| Reference: round-2 CatBoost (spend fill = 0) | **0.8169** | – | – |
| + group features | 0.8157 | −0.0012 | 2/5 |
| + cabin-neighborhood features | 0.8141 | −0.0029 | 0/5 |
| + group + neighborhood | 0.8158 | −0.0011 | 2/5 |
| + pseudo-labels (0.95/0.05; about 1,420 test rows per fold) | 0.8169 | 0.0000 | 1/5 |
| + pseudo-labels (0.90/0.10; about 1,980 test rows per fold) | 0.8156 | −0.0013 | 1/5 |

**What we learned**
1. **The group features only repeat what the model already knows.** Several correlate with the target as strongly as some existing features (`Grp_AgeGap` −0.13, `IsChild` +0.13, `Grp_SameCabin` +0.12), but that signal is already captured by `Age`, `GroupSize`, `CryoSleep` and the cabin features. Only `Nbr_CryoShare` reaches the top 15 in importance (#11), and adding it still lowers accuracy across seeds.
2. **Pseudo-labeling adds rows, not information.** The confident test rows are the cases the model already gets right (mostly sleepers on the high-rate decks). Retraining on them reinforces existing decision boundaries without helping the uncertain middle. The lower threshold brings in more mistakes and makes accuracy worse.
3. **Across three rounds, every lever has landed within about ±0.3 points of 0.8169.** That includes imputation variants, a target-based neighbor feature, three boosting libraries, ensembling, Optuna, group and neighborhood features, and pseudo-labeling. This is strong evidence that **about 0.817 grouped-CV accuracy is the limit of what this data supports**, and the remaining errors are label noise the features can't resolve (for example, sleepers on decks E and G at about 65% transported, or awake non-spenders at about 62%).

**Recommendation:** stop the hunt for accuracy. The round-1 CatBoost model (simplest, and as good as anything since) is the final model. Public scores above about 0.82 are mostly public-split noise, repeated submissions, or leaked labels.